<a href="https://colab.research.google.com/github/davis-mironga/marsabit-ecosystem-analysis/blob/main/01_GEE_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# PROJECT CONFIGURATION
# Edit these two lines if you are running this notebook on a different
# Google account or GEE project. Everything else in the notebook reads
# from these variables — you do not need to change anything else.
# ─────────────────────────────────────────────────────────────────────

GEE_PROJECT = 'mironga-project-marsabit'
ASSET_PATH  = 'projects/mironga-project-marsabit/assets/marsabit'

print(f'GEE project : {GEE_PROJECT}')
print(f'Asset path  : {ASSET_PATH}')


# Notebook 1 — GEE Data Preprocessing
## The Marsabit Footprint: A 34-Year Study on Livestock Pressure and Ecosystem Health

---

**Purpose**

This is the foundation notebook for the entire project. Everything produced here — NDVI layers, rainfall data, elevation, and the first livestock index component — feeds directly into Notebooks 2, 3, and 4.

All the satellite data is processed inside Google Earth Engine (GEE), which is Google's cloud platform for large-scale geospatial analysis. We do not download raw imagery. Instead, we write code that runs on Google's servers, processes hundreds of satellite images, and exports only the final clean layers we need.

**What this notebook produces**

| Output | Description |
|--------|-------------|
| NDVI composites | Vegetation index for 5 time periods (1990, 2000, 2010, 2020, 2024) |
| NDVI change maps | How vegetation changed between each period and over the full 34 years |
| CHIRPS rainfall | Mean annual rainfall per period — climate control variable |
| SRTM elevation | Terrain data — terrain control variable |
| LCI distance component | Distance-to-water layer — first part of the Livestock Concentration Index |

All 13 outputs are exported as GEE Assets for use in Notebooks 2, 3, and 4.

---

## Step 1 — Install Libraries and Authenticate with Google Earth Engine

This step installs the geemap library and connects the notebook to Google Earth Engine.

Two things happen here:

1. **geemap** is installed — a Python library that lets us run GEE commands and display interactive maps inside the notebook.
2. **GEE Authentication** — GEE needs to confirm you have permission to use its satellite data. When `ee.Authenticate()` runs, a link will appear. Click it, sign in with your GEE-registered Google account, and paste the code back into the box.

> When `ee.Authenticate()` runs, log in with the Google account registered for GEE access.


In [ ]:
# Install geemap (only needed once per Colab session)
!pip install geemap -q

import ee
import geemap

# Authenticate your Google account with GEE
# A link will appear — log in and paste the code back here
ee.Authenticate()

# Initialize GEE with your Google Cloud Project
ee.Initialize(project=GEE_PROJECT)

print(f"GEE authenticated. Project: {GEE_PROJECT}")

## Step 2 — Define the Study Area (Marsabit County Boundary)

This step loads the official boundary of Marsabit County from the FAO Global Administrative Unit Layers (GAUL) dataset, which is hosted directly inside GEE.

The boundary acts as a frame for everything that follows. Every satellite image we load will be clipped to this shape so we only work with data inside Marsabit County.

**Why FAO GAUL?**

It is an internationally recognised boundary dataset available directly inside GEE — no file uploads or local shapefiles needed.

**What the map shows**

An interactive map with the Marsabit County boundary outlined in red. This confirms we have the correct area before doing any analysis.

**Variables created**

| Variable | What it is |
|----------|------------|
| `marsabit_roi` | The full FeatureCollection — used for filtering satellite imagery |
| `marsabit_geom` | Just the geometry shape — used for clipping images to the county |


In [ ]:
# Load Marsabit County boundary from FAO GAUL Level 2 (district/county level)
marsabit_roi = (
    ee.FeatureCollection("FAO/GAUL/2015/level2")
    .filter(ee.Filter.eq('ADM2_NAME', 'Marsabit'))
)

# Extract just the geometry — used for clipping images later
marsabit_geom = marsabit_roi.geometry()

# Confirm it loaded correctly
area_km2 = marsabit_geom.area().divide(1e6).getInfo()
print(f"Marsabit County boundary loaded successfully.")
print(f"   Area: {area_km2:,.0f} km²")

# Visualize on an interactive map
Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map.centerObject(marsabit_roi, 8)
Map.addLayer(
    marsabit_roi,
    {'color': 'red'},
    'Marsabit County Boundary'
)
Map.addLayerControl()
Map

## Step 3 — Define the Satellite Image Loading Function

Here we write one reusable function that loads Landsat satellite imagery correctly for any time period. The function handles three important processing steps.

**1. Scaling**

Landsat Collection 2 Level 2 data stores pixel values as integers to save storage space. These integers are not meaningful on their own. We multiply by 0.0000275 and add -0.2 to convert them into real surface reflectance values (ranging from 0 to 1). Without this step, NDVI values would be scientifically incorrect.

**2. Cloud masking**

Every Landsat image comes with a quality band (QA_PIXEL) that flags cloudy pixels and cloud shadow pixels. We use this to automatically remove bad pixels before computing the composite. This ensures our NDVI represents actual vegetation, not cloud cover.

**3. Multi-year compositing**

Instead of using a single year, we work with 5-year windows (for example, 1988–1992 for the "1990" period). We take the median across all images in the window. The median is more robust than the mean — it removes remaining cloud artifacts and reduces the influence of unusual drought or flood years.

**The five time periods**

| Label | Date window | Satellite | Sensor code |
|-------|-------------|-----------|-------------|
| 1990 | 1988–1992 | Landsat 5 | L57 |
| 2000 | 1998–2002 | Landsat 5 | L57 |
| 2010 | 2008–2012 | Landsat 7 | L57 |
| 2020 | 2018–2022 | Landsat 8 | L89 |
| 2024 | 2022–2024 | Landsat 9 | L89 |

**Band note**

Landsat 5 and 7 use Band 4 (near-infrared) and Band 3 (red) to compute NDVI. Landsat 8 and 9 use Band 5 (near-infrared) and Band 4 (red). The function handles this automatically using the `sensor` parameter so we do not have to think about it when calling the function for each period.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# FUNCTION 1: Apply Landsat Collection 2 Level 2 Scale Factors
# Converts raw integer DN values real surface reflectance (0 to 1)
# ─────────────────────────────────────────────────────────────────────
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True)


# ─────────────────────────────────────────────────────────────────────
# FUNCTION 2: Cloud Mask using the QA_PIXEL band
# Removes cloud pixels (bit 5) and cloud shadow pixels (bit 3)
# ─────────────────────────────────────────────────────────────────────
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow = qa.bitwiseAnd(1 << 3).eq(0)  # Bit 3 = Cloud Shadow
    cloud        = qa.bitwiseAnd(1 << 5).eq(0)  # Bit 5 = Cloud
    return image.updateMask(cloud_shadow.And(cloud))


# ─────────────────────────────────────────────────────────────────────
# FUNCTION 3: Build NDVI Composite for a given time period
# This is the main function — we call it 5 times, once per decade
# ─────────────────────────────────────────────────────────────────────
def get_ndvi_composite(collection_id, start_year, end_year, roi, sensor='L57'):
    """
    Loads a multi-year Landsat median composite with:
    - Cloud masking applied
    - Scale factors applied
    - NDVI calculated and returned

    Args:
        collection_id : GEE Landsat collection string
        start_year    : Start year as string e.g. '1988'
        end_year      : End year as string e.g. '1992'
        roi           : GEE geometry to clip imagery to
        sensor        : 'L57' for Landsat 5 & 7 | 'L89' for Landsat 8 & 9

    Returns:
        Single-band NDVI image clipped to roi
    """
    collection = (
        ee.ImageCollection(collection_id)
        .filterBounds(roi)
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
        .map(mask_landsat_clouds)
        .map(apply_scale_factors)
    )

    composite = collection.median().clip(roi)

    # NDVI bands differ between sensor generations
    if sensor == 'L57':
        nir, red = 'SR_B4', 'SR_B3'   # Landsat 5 and 7
    else:
        nir, red = 'SR_B5', 'SR_B4'   # Landsat 8 and 9

    ndvi = composite.normalizedDifference([nir, red]).rename('NDVI')
    return ndvi


print("Three functions defined and ready:")
print("   apply_scale_factors()  : Raw DN surface reflectance")
print("   mask_landsat_clouds()  : Removes cloud & shadow pixels")
print("   get_ndvi_composite()   : Builds multi-year NDVI composite")

## Step 4 — Compute NDVI for All 5 Time Periods

We call the function from Step 3 five times — once per time period. Each call processes hundreds of satellite images automatically, applies cloud masking and scaling, and returns one clean NDVI image representing that era.

NDVI (Normalised Difference Vegetation Index) is a measure of how green and dense the vegetation is. It ranges from -1 to +1.

**How to read NDVI values**

| NDVI range | What it looks like on the ground |
|------------|----------------------------------|
| 0.6 – 1.0 | Dense, healthy vegetation — Mt. Marsabit forest |
| 0.3 – 0.6 | Moderate vegetation, typical rangeland |
| 0.1 – 0.3 | Sparse vegetation or degraded rangeland |
| 0.0 – 0.1 | Bare soil, rock, heavily degraded land |
| Below 0 | Water bodies |

**What the map shows**

All five NDVI layers loaded into one interactive map. Use the layer control panel in the top right to toggle each year on and off to compare vegetation across decades.

To see the clearest before-and-after picture: turn on 1990 and 2024, turn all others off.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Compute NDVI composites for all 5 time periods
# GEE processes these lazily — computation happens when the map renders
# ─────────────────────────────────────────────────────────────────────

print("Building NDVI composites for all 5 periods...")

ndvi_1990 = get_ndvi_composite(
    'LANDSAT/LT05/C02/T1_L2', '1988', '1992', marsabit_geom, sensor='L57'
)
print("   1990 period (1988–1992, Landsat 5)")

ndvi_2000 = get_ndvi_composite(
    'LANDSAT/LT05/C02/T1_L2', '1998', '2002', marsabit_geom, sensor='L57'
)
print("   2000 period (1998–2002, Landsat 5)")

ndvi_2010 = get_ndvi_composite(
    'LANDSAT/LE07/C02/T1_L2', '2008', '2012', marsabit_geom, sensor='L57'
)
print("   2010 period (2008–2012, Landsat 7)")

ndvi_2020 = get_ndvi_composite(
    'LANDSAT/LC08/C02/T1_L2', '2018', '2022', marsabit_geom, sensor='L89'
)
print("   2020 period (2018–2022, Landsat 8)")

ndvi_2024 = get_ndvi_composite(
    'LANDSAT/LC09/C02/T1_L2', '2022', '2024', marsabit_geom, sensor='L89'
)
print("   2024 period (2022–2024, Landsat 9)")

# ─────────────────────────────────────────────────────────────────────
# Visualization parameters
# Red = degraded/bare land | Green = healthy vegetation
# ─────────────────────────────────────────────────────────────────────
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['#d73027','#f46d43','#fdae61','#fee08b',
                '#d9ef8b','#a6d96a','#66bd63','#1a9850']
}

# Build the interactive map
Map2 = geemap.Map()
Map2.add_basemap("SATELLITE")
Map2.centerObject(marsabit_roi, 8)
Map2.addLayer(ndvi_1990, ndvi_vis, 'NDVI 1990')
Map2.addLayer(ndvi_2000, ndvi_vis, 'NDVI 2000', False)  # False = off by default
Map2.addLayer(ndvi_2010, ndvi_vis, 'NDVI 2010', False)
Map2.addLayer(ndvi_2020, ndvi_vis, 'NDVI 2020', False)
Map2.addLayer(ndvi_2024, ndvi_vis, 'NDVI 2024')
Map2.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map2.addLayerControl()

print("\nAll 5 NDVI composites ready!")
print("   Toggle layers on the map to compare decades.")
print("   Red = bare/degraded | Green = healthy vegetation")
Map2

## Step 5 — Compute NDVI Change Maps

Here we subtract NDVI values between time periods to create change maps. These show exactly where vegetation has increased (greening) or decreased (degradation) over each decade.

The formula is simple: Later NDVI minus Earlier NDVI. A positive result means vegetation gained; a negative result means it was lost.

**How to read the change maps**

| Colour | Value | Meaning |
|--------|-------|---------|
| Red | Negative | Vegetation loss — land became more bare |
| Yellow | Near zero | Little to no change |
| Green | Positive | Vegetation gain — land recovered |

**The four decadal change maps**

| Change map | Period | What it captures |
|------------|--------|-----------------|
| Change 1 | 1990 to 2000 | First decade |
| Change 2 | 2000 to 2010 | Second decade |
| Change 3 | 2010 to 2020 | Third decade |
| Change 4 | 2020 to 2024 | Most recent period |

**The most important output — Total Change (1990 to 2024)**

This is the full 34-year change map. It becomes the dependent variable Y in the regression model in Notebook 4:

```
NDVI_change = B0 + B1(LCI) + B2(Rainfall) + B3(Elevation) + error
```


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Compute NDVI change between each consecutive time period
# Formula: Change = Later period NDVI - Earlier period NDVI
# Positive = greening | Negative = degradation
# ─────────────────────────────────────────────────────────────────────

ndvi_change_90_00 = (ndvi_2000.subtract(ndvi_1990)
                    .rename('NDVI_change_1990_2000'))

ndvi_change_00_10 = (ndvi_2010.subtract(ndvi_2000)
                    .rename('NDVI_change_2000_2010'))

ndvi_change_10_20 = (ndvi_2020.subtract(ndvi_2010)
                    .rename('NDVI_change_2010_2020'))

ndvi_change_20_24 = (ndvi_2024.subtract(ndvi_2020)
                    .rename('NDVI_change_2020_2024'))

# Total change: 1990 2024 — this is our Y variable for regression
ndvi_change_total = (ndvi_2024.subtract(ndvi_1990)
                    .rename('NDVI_change_total_1990_2024'))

# ─────────────────────────────────────────────────────────────────────
# Visualization parameters for change maps
# Red = loss | Yellow = no change | Green = gain
# ─────────────────────────────────────────────────────────────────────
change_vis = {
    'min': -0.4,
    'max': 0.4,
    'palette': ['#d73027','#f46d43','#fdae61',
                '#ffffbf',
                '#a6d96a','#66bd63','#1a9850']
}

# ─────────────────────────────────────────────────────────────────────
# Visualize all change maps on an interactive map
# ─────────────────────────────────────────────────────────────────────
Map3 = geemap.Map()
Map3.add_basemap("SATELLITE")
Map3.centerObject(marsabit_roi, 8)

Map3.addLayer(ndvi_change_90_00, change_vis, 'Change 1990→2000')
Map3.addLayer(ndvi_change_00_10, change_vis, 'Change 2000→2010', False)
Map3.addLayer(ndvi_change_10_20, change_vis, 'Change 2010→2020', False)
Map3.addLayer(ndvi_change_20_24, change_vis, 'Change 2020→2024', False)
Map3.addLayer(ndvi_change_total, change_vis, 'TOTAL Change 1990→2024')
Map3.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map3.addLayerControl()

print("NDVI Change Maps computed:")
print("   ndvi_change_90_00  : Change 1990 to 2000")
print("   ndvi_change_00_10  : Change 2000 to 2010")
print("   ndvi_change_10_20  : Change 2010 to 2020")
print("   ndvi_change_20_24  : Change 2020 to 2024")
print("   ndvi_change_total  : TOTAL change 1990 to 2024 (Y variable)")
print("\nTip: Turn on TOTAL Change 1990→2024 layer for the full picture")
Map3

## Step 6 — Load CHIRPS Rainfall Data

This step loads rainfall data from CHIRPS (Climate Hazards Group InfraRed Precipitation with Station data). Rainfall is included in the regression model as a control variable.

**Why we need rainfall as a control**

Vegetation growth is heavily influenced by how much rain an area receives. Without controlling for rainfall, we cannot fairly isolate the effect of livestock pressure. An area might show low NDVI simply because it is dry — not because livestock grazed it heavily. By including rainfall in the regression, we separate what is explained by climate from what is explained by livestock.

**What CHIRPS is**

- Global rainfall dataset at 5 km resolution
- Available from 1981 to the present
- Combines satellite estimates with ground station records
- Widely used in dryland research across Africa

**What we compute**

We calculate mean annual rainfall for each of the five time periods, using the same date windows as the NDVI composites. We then compute the change in rainfall between 1990 and 2024. That change layer is what enters the regression model as X2.

**Processing steps applied**

1. Load CHIRPS daily data for each period
2. Sum daily values to get the total over the window
3. Divide by the number of years to get mean annual rainfall
4. Resample from 5 km to 30 m to match the Landsat resolution


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# FUNCTION: Compute mean annual rainfall for a given period
# CHIRPS data is daily — we sum to annual then average across years
# ─────────────────────────────────────────────────────────────────────
def get_chirps_rainfall(start_year, end_year, roi):
    """
    Computes mean annual rainfall (mm) for a multi-year period.

    Args:
        start_year : Start year as string e.g. '1988'
        end_year   : End year as string e.g. '1992'
        roi        : GEE geometry to clip to

    Returns:
        Single-band rainfall image clipped to roi (mean annual mm)
    """
    chirps = (
        ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
        .filterBounds(roi)
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
        .select('precipitation')
    )

    # Sum daily rainfall to get total for the whole period
    total_rainfall = chirps.sum()

    # Divide by number of years to get mean annual rainfall
    n_years = int(end_year) - int(start_year) + 1
    mean_annual = total_rainfall.divide(n_years)

    # Resample from 5km to 30m to match Landsat resolution
    mean_annual_30m = (mean_annual
                       .resample('bilinear')
                       .reproject(crs='EPSG:4326', scale=30))

    return mean_annual_30m.clip(roi).rename('rainfall_mm')


# ─────────────────────────────────────────────────────────────────────
# Compute rainfall for all 5 time periods
# Using same date windows as NDVI composites
# ─────────────────────────────────────────────────────────────────────
print("Computing CHIRPS rainfall for all 5 periods...")

rainfall_1990 = get_chirps_rainfall('1988', '1992', marsabit_geom)
print("   Rainfall 1990 period (1988–1992)")

rainfall_2000 = get_chirps_rainfall('1998', '2002', marsabit_geom)
print("   Rainfall 2000 period (1998–2002)")

rainfall_2010 = get_chirps_rainfall('2008', '2012', marsabit_geom)
print("   Rainfall 2010 period (2008–2012)")

rainfall_2020 = get_chirps_rainfall('2018', '2022', marsabit_geom)
print("   Rainfall 2020 period (2018–2022)")

rainfall_2024 = get_chirps_rainfall('2022', '2024', marsabit_geom)
print("   Rainfall 2024 period (2022–2024)")

# ─────────────────────────────────────────────────────────────────────
# Compute rainfall CHANGE between 1990 and 2024
# Used as control variable in regression
# ─────────────────────────────────────────────────────────────────────
rainfall_change = (rainfall_2024.subtract(rainfall_1990)
                  .rename('rainfall_change_1990_2024'))

# ─────────────────────────────────────────────────────────────────────
# Visualize rainfall on an interactive map
# Blue = high rainfall | White/Yellow = low rainfall
# ─────────────────────────────────────────────────────────────────────
rainfall_vis = {
    'min': 200,
    'max': 800,
    'palette': ['#ffffcc','#a1dab4','#41b6c4','#2c7fb8','#253494']
}

Map4 = geemap.Map()
Map4.add_basemap("SATELLITE")
Map4.centerObject(marsabit_roi, 8)
Map4.addLayer(rainfall_1990, rainfall_vis, 'Rainfall 1990')
Map4.addLayer(rainfall_2024, rainfall_vis, 'Rainfall 2024', False)
Map4.addLayer(marsabit_roi, {'color': 'red'}, 'Marsabit Boundary')
Map4.addLayerControl()

print("\nCHIRPS Rainfall layers ready!")
print("   rainfall_1990 to rainfall_2024  : Mean annual rainfall per period")
print("   rainfall_change                 : Rainfall change 1990→2024")
print("\nTip: Blue = high rainfall (Mt. Marsabit area)")
print("        Yellow = low rainfall (desert margins near Turkana)")
Map4

## Step 7 — Load SRTM Elevation Data

This step loads elevation data from the SRTM dataset (Shuttle Radar Topography Mission). Elevation is the second control variable in the regression model.

**Why elevation matters**

Elevation shapes vegetation in two direct ways:

1. Higher ground is cooler and receives more rainfall, which supports denser vegetation. Mt. Marsabit at around 1,700 m has montane forest because of this.
2. Lower ground is hotter and drier, supporting only sparse vegetation or desert conditions. The lowlands near Lake Turkana sit at around 400 m.

Without controlling for elevation, the regression might wrongly attribute to livestock the vegetation differences that are actually caused by terrain.

**What SRTM is**

- Captured by NASA Space Shuttle in February 2000
- Global coverage at 30 m resolution — the same as Landsat
- The standard elevation dataset used in GIS and remote sensing worldwide

**What we compute**

The raw elevation layer (in metres) for Marsabit County, clipped to the county boundary. We also derive slope from the elevation to provide additional terrain context. The elevation layer enters the regression model as X3.

**The full regression equation with all variables**

```
NDVI_change = B0 + B1(LCI) + B2(Rainfall) + B3(Elevation) + error
```


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Load SRTM Digital Elevation Model (DEM)
# Resolution: 30m — matches our Landsat imagery perfectly
# ─────────────────────────────────────────────────────────────────────
srtm = ee.Image("USGS/SRTMGL1_003")

# Clip to Marsabit boundary
elevation = srtm.select('elevation').clip(marsabit_geom)

# ─────────────────────────────────────────────────────────────────────
# Derive Slope from elevation
# Slope tells us terrain steepness — affects where livestock can graze
# ─────────────────────────────────────────────────────────────────────
slope = ee.Terrain.slope(srtm).clip(marsabit_geom)

# ─────────────────────────────────────────────────────────────────────
# Get basic elevation statistics for Marsabit
# ─────────────────────────────────────────────────────────────────────
stats = elevation.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        ee.Reducer.mean(), sharedInputs=True
    ),
    geometry=marsabit_geom,
    scale=30,
    maxPixels=1e9
)

stats_dict = stats.getInfo()
print("SRTM Elevation loaded successfully!")
print(f"   Elevation statistics for Marsabit County:")
print(f"   Minimum elevation : {stats_dict['elevation_min']:.0f} m")
print(f"   Maximum elevation : {stats_dict['elevation_max']:.0f} m")
print(f"   Mean elevation    : {stats_dict['elevation_mean']:.0f} m")

# ─────────────────────────────────────────────────────────────────────
# Visualization parameters
# ─────────────────────────────────────────────────────────────────────
elevation_vis = {
    'min': 300,
    'max': 1800,
    'palette': ['#313695','#4575b4','#74add1','#abd9e9',
                '#e0f3f8','#ffffbf','#fee090','#fdae61',
                '#f46d43','#d73027','#a50026']
}

slope_vis = {
    'min': 0,
    'max': 30,
    'palette': ['white', 'yellow', 'orange', 'red']
}

# ─────────────────────────────────────────────────────────────────────
# Visualize on interactive map
# ─────────────────────────────────────────────────────────────────────
Map5 = geemap.Map()
Map5.add_basemap("SATELLITE")
Map5.setCenter(37.9, 2.3, 8)
Map5.addLayer(elevation, elevation_vis, 'Elevation (m)')
Map5.addLayer(slope, slope_vis, 'Slope (degrees)', False)
Map5.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map5.addLayerControl()

print("\nTip: The bright peak in the centre is Mt. Marsabit (~1700m)")
print("        Dark blue = lowlands | Red/white = highlands")
Map5

## Step 8 — Build the Livestock Concentration Index (LCI) Distance Component

This step builds the first and most important component of the Livestock Concentration Index — the variable that measures how much livestock pressure each location experiences.

We do not have direct livestock count data at fine spatial resolution for Marsabit County. Instead, we use a spatial proxy: distance to water.

**The core idea**

In arid rangelands like Marsabit, livestock movement is driven almost entirely by water access. Animals congregate around boreholes, rivers, and pans, then spread outward as they graze. This creates a predictable pattern: the closer a location is to a water body, the more livestock pressure it experiences.

So instead of counting animals, we measure how far each pixel is from the nearest permanent water source and invert it. Pixels close to water get a high score; pixels far from water get a low score.

**The LCI formula (from the project methodology)**

```
LCI = α(1 / Distance_to_Water) + β(Settlement_Density)
```

In this notebook we compute the distance-to-water component. Settlement density is added in Notebook 3 to complete the full index.

**Steps in this calculation**

1. Load permanent water bodies from JRC Global Surface Water (pixels with water present more than 50% of the time)
2. Compute Euclidean distance from every pixel to the nearest water body (in metres)
3. Invert the distance: 1 / (distance + 1), so pixels near water score high
4. Normalise to a 0–1 scale

**Why this matters for the analysis**

If the regression in Notebook 4 finds that high-LCI areas have lower NDVI change, that is statistical evidence that livestock pressure — not just climate — is driving vegetation loss in Marsabit.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# STEP 8A: Load Water Bodies from JRC Global Surface Water dataset
# This gives us permanent water features — rivers, pans, lakes
# These are the places livestock concentrate around
# ─────────────────────────────────────────────────────────────────────

# Load JRC Global Surface Water — occurrence band (0-100%)
# We keep pixels with water occurrence > 50% (permanent/semi-permanent water)
jrc_water = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select('occurrence')
    .gte(50)  # Keep pixels that are water at least 50% of the time
    .clip(marsabit_geom)
)

# Convert water pixels to vectors (points) for distance calculation
water_vectors = jrc_water.selfMask().reduceToVectors(
    geometry=marsabit_geom,
    scale=500,           # 500m scale for efficiency
    maxPixels=1e9,
    geometryType='centroid'
)

print(f"Water features loaded")
print(f"   Water bodies found: {water_vectors.size().getInfo()} features")

# ─────────────────────────────────────────────────────────────────────
# STEP 8B: Compute Distance to Nearest Water Point
# For every pixel in Marsabit, calculate distance to nearest water body
# ─────────────────────────────────────────────────────────────────────

# Distance transform — every pixel gets distance to nearest water (metres)
distance_to_water = (
    jrc_water.fastDistanceTransform(neighborhood=2048)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())  # Convert to metres
    .clip(marsabit_geom)
    .rename('distance_to_water_m')
)

# ─────────────────────────────────────────────────────────────────────
# STEP 8C: Build the LCI Distance Component
# Invert distance: pixels NEAR water = HIGH livestock pressure
# Apply min-max normalization to scale between 0 and 1
# ─────────────────────────────────────────────────────────────────────

# Invert: 1/distance (add small value to avoid division by zero)
inverse_distance = (
    ee.Image(1)
    .divide(distance_to_water.add(1))
    .rename('inverse_distance')
)

# Min-max normalization to 0-1 scale
dist_min_max = inverse_distance.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=marsabit_geom,
    scale=500,
    maxPixels=1e9
)

dist_min = ee.Number(dist_min_max.get('inverse_distance_min'))
dist_max = ee.Number(dist_min_max.get('inverse_distance_max'))

# Normalized LCI distance component (0 = far from water, 1 = at water)
lci_distance = (
    inverse_distance.subtract(dist_min)
    .divide(dist_max.subtract(dist_min))
    .rename('LCI_distance_component')
)

# ─────────────────────────────────────────────────────────────────────
# STEP 8D: Visualize Distance to Water and LCI component
# ─────────────────────────────────────────────────────────────────────
distance_vis = {
    'min': 0,
    'max': 50000,  # 0 to 50km
    'palette': ['#08306b','#2171b5','#6baed6','#bdd7e7','#eff3ff','white']
}

lci_vis = {
    'min': 0,
    'max': 1,
    'palette': ['white', '#ffffb2', '#fecc5c', '#fd8d3c', '#e31a1c']
}

Map6 = geemap.Map()
Map6.add_basemap("SATELLITE")
Map6.setCenter(37.9, 2.3, 8)
Map6.addLayer(distance_to_water, distance_vis, 'Distance to Water (m)')
Map6.addLayer(lci_distance, lci_vis, 'LCI Distance Component', False)
Map6.addLayer(jrc_water.selfMask(),
              {'palette': ['blue']}, 'Water Bodies')
Map6.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map6.addLayerControl()

print("Livestock Concentration Index — Distance Component ready!")
print("   distance_to_water  : Distance in metres to nearest water body")
print("   lci_distance       : Normalized LCI (0=far from water, 1=at water)")
print("\nOn the LCI map:")
print("   Red/Orange = HIGH livestock pressure (near water)")
print("   White      = LOW livestock pressure (far from water)")
Map6

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Export configuration — GEE Assets
# ─────────────────────────────────────────────────────────────────────
# ASSET_PATH is defined in the configuration cell at the top of the notebook
SCALE = 30
CRS = 'EPSG:4326'

# ─────────────────────────────────────────────────────────────────────
# STEP 9A: Create the asset folder first
# Run this ONCE — comment it out after first run
# ─────────────────────────────────────────────────────────────────────
try:
    ee.data.createAsset(
        {'type': 'Folder'},
        ASSET_PATH
    )
    print(f"Asset folder created: {ASSET_PATH}")
except Exception as e:
    print(f" Folder may already exist: {e}")

# ─────────────────────────────────────────────────────────────────────
# HELPER FUNCTION: Export image to GEE Asset
# ─────────────────────────────────────────────────────────────────────
def export_to_asset(image, asset_name, description):
    """
    Submits a GEE export task to GEE Assets.

    Args:
        image      : GEE image to export
        asset_name : Asset filename (no path needed)
        description: Task description in GEE Task Manager
    """
    asset_id = f'{ASSET_PATH}/{asset_name}'

    task = ee.batch.Export.image.toAsset(
        image=image,
        description=description,
        assetId=asset_id,
        region=marsabit_geom,
        scale=SCALE,
        crs=CRS,
        maxPixels=1e13
    )
    task.start()
    return task


# ─────────────────────────────────────────────────────────────────────
# Submit all export tasks to GEE Assets
# ─────────────────────────────────────────────────────────────────────
print("Submitting export tasks to GEE Assets...")
print(f"   Path: {ASSET_PATH}/\n")

# --- NDVI Composites ---
export_to_asset(ndvi_1990, 'ndvi_1990', 'NDVI_1990_Marsabit')
print("   Task submitted: ndvi_1990")

export_to_asset(ndvi_2000, 'ndvi_2000', 'NDVI_2000_Marsabit')
print("   Task submitted: ndvi_2000")

export_to_asset(ndvi_2010, 'ndvi_2010', 'NDVI_2010_Marsabit')
print("   Task submitted: ndvi_2010")

export_to_asset(ndvi_2020, 'ndvi_2020', 'NDVI_2020_Marsabit')
print("   Task submitted: ndvi_2020")

export_to_asset(ndvi_2024, 'ndvi_2024', 'NDVI_2024_Marsabit')
print("   Task submitted: ndvi_2024")

# --- NDVI Change Maps ---
export_to_asset(ndvi_change_90_00,
                'ndvi_change_1990_2000', 'NDVI_Change_1990_2000')
print("   Task submitted: ndvi_change_1990_2000")

export_to_asset(ndvi_change_00_10,
                'ndvi_change_2000_2010', 'NDVI_Change_2000_2010')
print("   Task submitted: ndvi_change_2000_2010")

export_to_asset(ndvi_change_10_20,
                'ndvi_change_2010_2020', 'NDVI_Change_2010_2020')
print("   Task submitted: ndvi_change_2010_2020")

export_to_asset(ndvi_change_20_24,
                'ndvi_change_2020_2024', 'NDVI_Change_2020_2024')
print("   Task submitted: ndvi_change_2020_2024")

export_to_asset(ndvi_change_total,
                'ndvi_change_total', 'NDVI_Change_Total_1990_2024')
print("   Task submitted: ndvi_change_total")

# --- Control Variables ---
export_to_asset(rainfall_change,
                'rainfall_change', 'Rainfall_Change_1990_2024')
print("   Task submitted: rainfall_change")

export_to_asset(elevation,
                'elevation', 'Elevation_SRTM')
print("   Task submitted: elevation")

export_to_asset(lci_distance,
                'lci_distance', 'LCI_Distance_Component')
print("   Task submitted: lci_distance")

# ─────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────
print("\nAll 13 export tasks submitted to GEE Assets!")
print("─────────────────────────────────────────────────")
print("Next steps:")
print("   1. Go to code.earthengine.google.com")
print("   2. Click the 'Tasks' tab on the right panel")
print("   3. Click RUN on any UNSUBMITTED tasks")
print("   4. Wait for all tasks COMPLETED (5–30 mins)")
print("   5. Check assets at:")
print(f"      {ASSET_PATH}/")
print("─────────────────────────────────────────────────")
print(" Keep this Colab tab open until tasks are RUNNING")

In [ ]:
# Run this cell to confirm which GEE project your account is connected to.
# Useful when switching between accounts or troubleshooting access issues.
# Check your personal GEE username
username = ee.data.getAssetRoots()
print("Your GEE asset roots:")
for root in username:
    print(f"   {root['id']}")

In [ ]:
# Verify all assets were exported successfully
# ASSET_PATH is defined in the configuration cell at the top

assets = ee.data.listAssets({'parent': ASSET_PATH})
asset_list = assets.get('assets', [])

expected = [
    'ndvi_1990', 'ndvi_2000', 'ndvi_2010', 'ndvi_2020', 'ndvi_2024',
    'ndvi_change_1990_2000', 'ndvi_change_2000_2010',
    'ndvi_change_2010_2020', 'ndvi_change_2020_2024',
    'ndvi_change_total', 'rainfall_change', 'elevation', 'lci_distance'
]

print(f"Assets found in GEE: {len(asset_list)}/13")
print("─────────────────────────────────────────")

found_names = [a['name'].split('/')[-1] for a in asset_list]

for name in expected:
    if name in found_names:
        print(f"   {name}")
    else:
        print(f"   {name} — NOT YET EXPORTED")

print("─────────────────────────────────────────")
if len(asset_list) == 13:
    print("All 13 assets exported successfully!")
    print("   Notebook 1 is COMPLETE!")
else:
    print(f"Still waiting — {13 - len(asset_list)} assets remaining")

## Notebook 1 — Summary

All preprocessing steps completed successfully.

---

### Steps completed

| Step | Task | Status |
|------|------|--------|
| 1 | GEE authentication and setup | Done |
| 2 | Marsabit County boundary loaded | Done |
| 3 | Satellite image functions defined | Done |
| 4 | NDVI composites for 5 time periods | Done |
| 5 | NDVI change maps (decadal + total) | Done |
| 6 | CHIRPS rainfall data | Done |
| 7 | SRTM elevation data | Done |
| 8 | LCI distance-to-water component | Done |
| 9 | All 13 assets exported to GEE | Done |

---

### All 13 assets stored at

the asset path defined in the configuration cell at the top of this notebook

| Asset | Description |
|-------|-------------|
| ndvi_1990 to ndvi_2024 (x5) | NDVI composites for each time period |
| ndvi_change_1990_2000 to ndvi_change_2020_2024 (x4) | Decadal NDVI change maps |
| ndvi_change_total | Total NDVI change 1990 to 2024 — regression Y variable |
| rainfall_change | Rainfall change 1990 to 2024 — regression X2 |
| elevation | SRTM elevation — regression X3 |
| lci_distance | LCI distance-to-water component |

---

### Next: Notebook 2 — LULC Classification

Random Forest land cover classification for all five time periods (1990, 2000, 2010, 2020, 2024).
